# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully referenced via Croissant `@id` fields, ensuring precise and reproducible data access and processing.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access high-level metadata (see documentation for full attributes)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their fields and `@id`s to understand the data structure.

We'll enumerate record sets, fields and their columns as defined by their `@id` attributes.

In [ ]:
print("Available record sets and their fields:\n")

# Croissant allows access to all record sets defined
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in dataset; please check dataset schema.")
else:
    for rs in record_sets:
        print(f"RecordSet name    : {rs.name}")
        print(f"  @id             : {rs.id}")
        print(f"  Description     : {rs.description if hasattr(rs,'description') else 'N/A'}\n")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field name: {getattr(field,'name',None)}")
            print(f"      @id      : {getattr(field, 'id', None)}")
            col_ids = getattr(field, 'columns', [])
            if col_ids:
                # Some schemas provide columns as Croissant Column objects, some just as @ids
                col_ids = [c.id if hasattr(c,'id') else str(c) for c in col_ids]
            print(f"      columns  : {col_ids}")
            print(f"      type     : {getattr(field,'data_type',None)}\n")
        print("-"*50)

## 3. Data Extraction
We will extract data from available record sets using their `@id` and load records into pandas DataFrames. Each record set and its fields are referenced by their `@id` fields, as required for reproducibility.

In [ ]:
# List of available record sets (by `@id`) from the overview above:
record_sets = [rs.id for rs in dataset.record_sets()]
print(f"Record set @ids: {record_sets}\n")

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))  # Returns list of Python dict
    if records:
        df = pd.DataFrame(records)
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        dataframes[record_set_id] = df
        display(df.head())
    else:
        print(f"No records found for {record_set_id}\n")

## 4. Exploratory Data Analysis (EDA)
This section explores numeric fields in one of the record sets, applies filters, normalization, and simple grouping.

**Please update `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` based on the discovered field `@id`s above.**

Below is an example EDA process for the first available record set, using its column `@id`s. All identifiers use Croissant `@id` notation for portability.

In [ ]:
# For this example, let's use the first non-empty record set for EDA
if not dataframes:
    print("No loaded dataframes available for EDA.")
else:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    print(f"Available columns (@id): {df.columns.tolist()}")
    # Try to select a numeric field (float or int)
    # Example: the field describing protein abundance, such as 'abundance' or 'MW' or similar
    # In actual usage, replace below with the exact field @id determined from overview above
    numeric_field_candidates = [c for c in df.columns if df[c].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_field_candidates:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = numeric_field_candidates[0]  # Use the first numeric field
        print(f"Using numeric field (@id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in [float, int] else 10  # fallback threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by another field
        group_field_candidates = [c for c in df.columns if c != numeric_field_id]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"\nGrouping by field (@id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped DataFrame by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No group field available for grouping.")

## 5. Visualization

We can visualize the distribution of our chosen numeric field, and—for example—compare grouped mean values if possible.

**NOTE:** Replace plot variable assignments with the actual field `@id`s as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_set_id]
    # Check if numeric_field_id was selected
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(10,5))
        sns.histplot(df[numeric_field_id], kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()
        if 'group_field' in locals():
            plt.figure(figsize=(12,6))
            grouped_mean = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
            sns.barplot(x=grouped_mean.index[:10], y=grouped_mean.values[:10])
            plt.title(f"Top 10 mean {numeric_field_id} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("Numeric field not selected for visualization.")

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset via its Croissant schema, explored its structure, and demonstrated field-level data access using entity `@id`. We previewed the available data, performed simple filtering, normalization, and grouping operations for basic EDA. Finally, we visualized field distributions and group comparisons.

**Best practices:**
- Always reference record sets, fields, columns, and data types by their Croissant `@id` to ensure reproducible and schema-agnostic code.
- For advanced analytics or modeling, validate the field types (string vs. float, etc.) and consult the schema documentation for canonical field meanings.

Explore further by adapting the code to additional record sets, fields, or linking to documentation referenced by Croissant metadata. If you need deeper access to the schema or column definitions, see `dataset.metadata.to_json()` or inspect objects directly.